# Full training pipeline — nfft512_hop128

Runs stages 00–05 in dependency order: intake → spectrograms → baseline → denoise → labels → dataset → refine → final model → TJ-II evaluation. Each CPU/GPU job must be COMPLETED and recorded in the task matrix before the next stage starts.

This is a non-interactive run. Review galleries in the individual notebooks before a production run; this notebook bypasses their human sign-off gate only after the runner artifact-completion gate has passed.

In [ ]:
from hashlib import sha256

from tokeye.training.big_tf_unet_2 import slurm
from tokeye.training.big_tf_unet_2.notebook_api import Run

RUN_ID = "nfft512_hop128"
POLL_SECONDS = 30
run = Run.open(RUN_ID)
print(f"run.yaml sha256: {sha256(run.paths.run_yaml.read_bytes()).hexdigest()}")
run.status()

In [ ]:
# Resume-safe helpers. Do not call run.submit(...) directly below.
ACTIVE_SLURM_STATES = {"PENDING", "RUNNING", "CONFIGURING", "COMPLETING"}

def is_complete(step):
    return run._matrix.all_complete(step, run.cfg.modality_names)

def run_inline(step):
    if is_complete(step):
        print(f"{step}: already complete; preserving checkpoint")
        return
    print(f"{step}: running inline")
    run.run(step, force=True)
    if not is_complete(step):
        raise RuntimeError(f"{step} finished without all required artifacts")

def run_cluster(step):
    if is_complete(step):
        print(f"{step}: already complete; preserving checkpoint")
        return
    matrix = run._matrix
    modalities = run.cfg.modality_names if step in {"step_2", "step_3"} else [None]
    job_ids = {matrix.job_id(step, mod) for mod in modalities}
    job_ids.discard(None)
    for job_id in job_ids:
        state = slurm.job_state(str(job_id))
        if state in ACTIVE_SLURM_STATES:
            print(f"{step}: waiting for existing job {job_id}")
            state = run.wait(str(job_id), poll=POLL_SECONDS)
        if state == "COMPLETED" and is_complete(step):
            print(f"{step}: existing job {job_id} completed")
            return
        print(f"{step}: prior job {job_id} ended as {state}; clearing partial artifacts")
        run.clear(step)
        break
    job_id = run.submit(step)
    state = run.wait(job_id, poll=POLL_SECONDS)
    if state != "COMPLETED" or not is_complete(step):
        run.log(job_id)
        raise RuntimeError(f"{step} job {job_id} ended as {state}; pipeline stopped safely")
    print(f"{step}: job {job_id} completed")

## 00–01 — Setup, intake, spectrograms, baseline

step_0 extracts each diagnostic into windows; step_1 creates STFTs and filters inactive windows; step_2 removes smooth background on a CPU node.

In [ ]:
run_inline("step_0")
run.status()

In [ ]:
# View raw, windowed signals for every modality. This never submits a job.
run.gallery("step_0", n=2)

In [ ]:
run_inline("step_1")
run.status()

In [ ]:
# Spectrograms after window filtering (one gallery per ECE, MHR, and BES).
run.gallery("step_1", n=3)

In [ ]:
run_cluster("step_2")
run.status()

In [ ]:
# Baseline removal: left is the input spectrogram; right is the residual.
run.gallery("step_2", n=3)

## 02–03 — Denoise and label

step_3 trains one self-supervised cross-channel denoiser per modality on a GPU. step_4 converts residual fields to coherent/transient masks.

In [ ]:
run_cluster("step_3")
run.status()

In [ ]:
# Denoising comparison: residual input beside the denoised spectrogram.
run.gallery("step_3", n=3)

In [ ]:
run_inline("step_4")
run.status()

In [ ]:
# Masks and automatically selected thresholds for each modality.
run.gallery("step_4", n=3)

## 04 — Build, refine, and export the model

step_5 combines examples. step_6 makes out-of-fold GPU predictions to refine masks. step_7 trains and exports the final TorchScript model.

In [ ]:
run_inline("step_5")
run.status()

In [ ]:
run.gallery("step_5", n=6)

In [ ]:
run_cluster("step_6")
run.status()

In [ ]:
run.gallery("step_6", n=6)

In [ ]:
run_cluster("step_7")
run.status()

In [ ]:
run.gallery("step_7")

## 05 — Evaluate

step_8 runs the exported model on TJ-II and writes data/cache/big_tf_unet_2/nfft512_hop128/eval_tjii.csv.

In [ ]:
run_inline("step_8")
run.status()

In [ ]:
run.gallery("step_8")

## Modality routing

- **ECE** is electron-cyclotron-emission data (channels 8–36); steps 0–4, then step 5.
- **MHR** is magnetic high-resolution data (channels 3–6); steps 0–4, then step 5.
- **BES** is beam-emission spectroscopy data (channels 26–40); steps 0–4, then step 5.

Steps 0–4 run once per diagnostic. Step 5 merges all three; steps 6–8 operate on the merged dataset and final model.